# 1) Parse hierarchy text into Superclass–Subclass CSV format

In [1]:
import pandas as pd
import os

def parse_hierarchy_to_csv(input_file_path, output_file_path="DILON_2.csv"):
    """
    Read a hierarchy file and convert it into superclass–subclass relationships,
    then save the result as a CSV file.

    Args:
        input_file_path (str): Path to the input file.
        output_file_path (str): Path to the output CSV file.
    """
    
    # Check if file exists
    if not os.path.exists(input_file_path):
        print(f"File not found: {input_file_path}")
        print(f"Current working directory: {os.getcwd()}")
        return
    
    # Read file
    try:
        if input_file_path.endswith('.xlsx'):
            df = pd.read_excel(input_file_path)  # Excel files are read with read_excel
        elif input_file_path.endswith('.csv'):
            df = pd.read_csv(input_file_path)
        else:
            # Treat as plain text file
            with open(input_file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            df = pd.DataFrame({'content': [line.strip() for line in lines if line.strip()]})
        
        print(f"File loaded successfully: {len(df)} rows")
        
    except Exception as e:
        print(f"Failed to read file: {e}")
        return
    
    # List to store all relationships
    relationships = []
    
    # Scan all cells and parse hierarchy expressions
    for col in df.columns:
        for value in df[col].dropna():
            value_str = str(value).strip()
            
            # Handle lines containing '>' as hierarchy expressions
            if '>' in value_str and value_str.lower() != 'hierarchy':
                concepts = [concept.strip() for concept in value_str.split('>')]
                
                # Create superclass–subclass relationships for consecutive concepts
                for i in range(len(concepts) - 1):
                    superclass = concepts[i]
                    subclass = concepts[i + 1]
                    
                    if superclass and subclass:
                        relationships.append([superclass, subclass])
    
    print(f"Extracted relationships: {len(relationships)}")
    
    # Remove duplicates
    unique_relationships = []
    seen = set()
    for rel in relationships:
        rel_tuple = tuple(rel)
        if rel_tuple not in seen:
            seen.add(rel_tuple)
            unique_relationships.append(rel)
    
    print(f"After removing duplicates: {len(unique_relationships)}")
    
    # Convert to DataFrame and save as CSV
    result_df = pd.DataFrame(unique_relationships, columns=['Superclass', 'Subclass'])
    result_df.to_csv(output_file_path, index=False)
    
    print(f"Results saved to: {output_file_path}")
    print("First 10 relationships:")
    print(result_df.head(10))


# Run - update with full paths as needed
file_path = "/Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/final_with_n_prefix.txt"
output_path = "/Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_2.csv"

print(f"Input file: {file_path}")
print(f"Output file: {output_path}")

parse_hierarchy_to_csv(file_path, output_path)


Input file: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/final_with_n_prefix.txt
Output file: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_2.csv
File loaded successfully: 202 rows
Extracted relationships: 855
After removing duplicates: 294
Results saved to: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_2.csv
First 10 relationships:
               Superclass                Subclass
0        Dietary_ontology              Descriptor
1              Descriptor        Measurement_unit
2        Measurement_unit  X_Quantity_measurement
3  X_Quantity_measurement                N_Amount
4        Dietary_ontology               Food_item
5               Food_item          Raw_ingredient
6          Raw_ingredient                 Seafood
7                 Seafood                    Fish
8                    Fish               N_Anchovy
9        Dietary_ontology         Personal_factor


# 2. Merge two DILON CSV files and remove duplicates

In [6]:
import pandas as pd
import os

def merge_dilon_csv(file1_path, file2_path, output_path):
    """
    Merge two DILON CSV files and remove duplicates.
    
    Args:
        file1_path (str): Path to the first CSV file.
        file2_path (str): Path to the second CSV file. 
        output_path (str): Path to save the merged CSV file.
    """
    
    print(f"File 1: {file1_path}")
    print(f"File 2: {file2_path}")
    print(f"Output file: {output_path}")
    
    # Check file existence
    if not os.path.exists(file1_path):
        print(f"File 1 not found: {file1_path}")
        return
    
    if not os.path.exists(file2_path):
        print(f"File 2 not found: {file2_path}")
        return
    
    try:
        # Read CSV files
        df1 = pd.read_csv(file1_path)
        df2 = pd.read_csv(file2_path)
        
        print(f"File 1 loaded successfully: {len(df1)} relationships")
        print(f"File 2 loaded successfully: {len(df2)} relationships")
        
        # Print column names
        print(f"File 1 columns: {list(df1.columns)}")
        print(f"File 2 columns: {list(df2.columns)}")
        
        # Merge dataframes
        merged_df = pd.concat([df1, df2], ignore_index=True)
        print(f"After merging: {len(merged_df)} relationships")
        
        # Remove duplicates
        merged_df = merged_df.drop_duplicates()
        print(f"After removing duplicates: {len(merged_df)} relationships")
        
        # Optional: sort by columns
        merged_df = merged_df.sort_values(['Superclass', 'Subclass'])
        
        # Save result
        merged_df.to_csv(output_path, index=False)
        print(f"Merged result saved to: {output_path}")
        
        # Statistics
        unique_superclasses = merged_df['Superclass'].nunique()
        unique_subclasses = merged_df['Subclass'].nunique()
        
        print("\n=== Statistics ===")
        print(f"Unique superclasses: {unique_superclasses}")
        print(f"Unique subclasses: {unique_subclasses}")
        print(f"Total relationships: {len(merged_df)}")
        
        # Preview
        print("\n=== Preview (first 15 rows) ===")
        print(merged_df.head(15).to_string(index=False))
        
        return merged_df
        
    except Exception as e:
        print(f"Failed to process files: {e}")
        return None

# File paths
base_path = "/Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2"
file1_path = f"{base_path}/DILON_1.csv"
file2_path = f"{base_path}/DILON_2.csv"
output_path = f"{base_path}/DILON_merged.csv"

# Run
print("Starting DILON CSV merge!")
merged_result = merge_dilon_csv(file1_path, file2_path, output_path)

if merged_result is not None:
    print("Task completed!")
else:
    print("Task failed!")


Starting DILON CSV merge!
File 1: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_1.csv
File 2: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_2.csv
Output file: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_merged.csv
File 1 loaded successfully: 255 relationships
File 2 loaded successfully: 294 relationships
File 1 columns: ['Superclass', 'Subclass']
File 2 columns: ['Superclass', 'Subclass']
After merging: 549 relationships
After removing duplicates: 491 relationships
Merged result saved to: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_merged.csv

=== Statistics ===
Unique superclasses: 137
Unique subclasses: 488
Total relationships: 491

=== Preview (first 15 rows) ===
     Superclass           Subclass
Allergy_symptom   Hypersensitivity
Allergy_symptom           Pruritus
Allergy_symptom               Rash
    Bakery_food    X_Baked_dessert
       Beverage  Caffeinated_d

# 3-1. Expand the merged DILON CSV into a hierarchical CSV structure

In [7]:
import pandas as pd
from collections import defaultdict
import os
import math

def expand_dilon_hierarchy_fixed(input_csv_path, output_csv_path):
    """
    Expand the DILON merged CSV into full hierarchical paths.
    Input: CSV with columns Superclass, Subclass
    Output: CSV with Level_1, Level_2, ... representing full root-to-leaf paths
    """
    print(f"Input file: {input_csv_path}")
    print(f"Output file: {output_csv_path}")
    
    if not os.path.exists(input_csv_path):
        print(f"Input file not found: {input_csv_path}")
        return None
    
    try:
        # 1) Load CSV
        df = pd.read_csv(input_csv_path)
        print(f"CSV loaded successfully: {len(df)} relationships")

        # Defensive column handling
        cols = {c.strip().lower(): c for c in df.columns}
        if "superclass" not in cols or "subclass" not in cols:
            raise ValueError(f"The input CSV must contain 'Superclass' and 'Subclass' columns. Found: {list(df.columns)}")

        col_super = cols["superclass"]
        col_sub   = cols["subclass"]

        # 2) Build relationships (remove blanks, NaN-like values)
        children_map = defaultdict(list)
        parents = {}
        all_nodes = set()

        def _is_blank(x):
            if x is None:
                return True
            if isinstance(x, float) and math.isnan(x):
                return True
            s = str(x).strip()
            return (s == "") or (s.lower() == "nan")

        clean_rows = 0
        for _, row in df.iterrows():
            superclass = str(row[col_super]).strip()
            subclass   = str(row[col_sub]).strip()
            if _is_blank(superclass) or _is_blank(subclass):
                continue
            children_map[superclass].append(subclass)
            parents[subclass] = superclass
            all_nodes.add(superclass)
            all_nodes.add(subclass)
            clean_rows += 1

        print(f"Valid relationships (after cleaning): {clean_rows}")
        print(f"Total nodes: {len(all_nodes)}")

        # 3) Root nodes = nodes that do not appear as children
        true_roots = [n for n in all_nodes if n not in parents]
        print(f"Root nodes ({len(true_roots)}): {true_roots[:10]}{' ...' if len(true_roots) > 10 else ''}")

        # 4) Validate missing parents
        print("\n=== Data Validation ===")
        orphan_nodes = []
        for child, parent in parents.items():
            if parent not in all_nodes:
                orphan_nodes.append((child, parent))
        if orphan_nodes:
            print(f"Missing parent relationships: {len(orphan_nodes)} (showing top 10):")
            for child, missing_parent in orphan_nodes[:10]:
                print(f"  {child} -> {missing_parent} (missing)")
        else:
            print("No missing parent relationships")

        # 5) Get full path from root to a given node
        def get_full_path_to_root(node):
            path = [node]
            current = node
            visited = set()
            while current in parents and current not in visited:
                visited.add(current)
                current = parents[current]
                path.append(current)
            return path[::-1]

        # 6) Leaf nodes = nodes with no children
        leaf_nodes = [n for n in all_nodes if (n not in children_map or len(children_map[n]) == 0)]
        print(f"\nLeaf nodes: {len(leaf_nodes)}")

        # 7) Collect all leaf paths & remove duplicates
        all_paths = [get_full_path_to_root(leaf) for leaf in leaf_nodes]
        unique_paths = []
        seen = set()
        for p in all_paths:
            t = tuple(p)
            if t not in seen:
                seen.add(t)
                unique_paths.append(p)

        print(f"Unique paths collected: {len(unique_paths)}")

        if not unique_paths:
            print("No hierarchical paths found.")
            return None

        # 8) Determine depth and build final table
        max_depth = max(len(p) for p in unique_paths)
        min_depth = min(len(p) for p in unique_paths)
        print(f"Max depth: {max_depth}, Min depth: {min_depth}")

        rows = []
        for p in unique_paths:
            padded = p + [None] * (max_depth - len(p))
            rows.append(padded)

        df_expanded = pd.DataFrame(rows, columns=[f"Level_{i+1}" for i in range(max_depth)])

        # 9) Preview
        print("\n=== Preview ===")
        print(f"Generated DataFrame: {len(df_expanded)} rows x {df_expanded.shape[1]} columns")
        print(df_expanded.head(10).to_string(index=False))

        # 10) Save CSV (UTF-8 with BOM for Excel compatibility)
        df_expanded.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
        print(f"\nCSV saved successfully: {output_csv_path}")

        # 11) Level analysis
        print("\n=== Level-wise Analysis ===")
        for col in df_expanded.columns:
            uniq = df_expanded[col].dropna().unique()
            print(f"{col}: {len(uniq)} unique nodes", end="")
            if len(uniq) <= 10:
                print(f" → {list(uniq)}")
            else:
                print()

        return df_expanded

    except Exception as e:
        print(f"Error occurred during processing: {e}")
        import traceback
        traceback.print_exc()
        return None


# ===== Execution =====
base_path = "/Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2"
input_csv = f"{base_path}/DILON_merged.csv"
output_csv = f"{base_path}/DILON_merged_with_hierarchy.csv"

print("Starting DILON hierarchy expansion!")
result = expand_dilon_hierarchy_fixed(input_csv, output_csv)

if result is not None:
    print("\nTask completed!")
    print("Example output: Dietary_ontology > Food_item > Beverage > ...")
else:
    print("Task failed!")


Starting DILON hierarchy expansion!
Input file: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_merged.csv
Output file: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_merged_with_hierarchy.csv
CSV loaded successfully: 491 relationships
Valid relationships (after cleaning): 491
Total nodes: 489
Root nodes (1): ['Dietary_ontology']

=== Data Validation ===
No missing parent relationships

Leaf nodes: 352
Unique paths collected: 352
Max depth: 7, Min depth: 3

=== Preview ===
Generated DataFrame: 352 rows x 7 columns
         Level_1              Level_2                  Level_3                Level_4                       Level_5 Level_6 Level_7
Dietary_ontology           Descriptor                Qualifier                Pattern                       Regular    None    None
Dietary_ontology      Personal_factor         Physical_finding        Allergy_symptom                          Rash    None    None
Dietary_ontology Food_handlin

# 3-2. Convert the merged DILON CSV into a hierarchical OWL structure

In [13]:
#!/usr/bin/env python
# coding: utf-8

import pandas as pd
from collections import defaultdict
import os
import math
import re
from pathlib import Path
from rdflib import Graph  # for converting OWL (RDF/XML) → Turtle

def expand_dilon_hierarchy_fixed(input_csv_path, output_csv_path):
    """
    Expand a DILON merged CSV (columns: Superclass, Subclass) into a full hierarchy.
    Output a CSV containing columns Level_1..Level_N (root → leaf paths).
    """
    print(f"Input file: {input_csv_path}")
    print(f"Output file: {output_csv_path}")
    
    if not os.path.exists(input_csv_path):
        print(f"Input file not found: {input_csv_path}")
        return None
    
    try:
        # 1) Load CSV
        df = pd.read_csv(input_csv_path)
        print(f"CSV loaded successfully: {len(df)} relationships")

        # Defensive column handling
        cols = {c.strip().lower(): c for c in df.columns}
        if "superclass" not in cols or "subclass" not in cols:
            raise ValueError(
                f"The CSV must contain 'Superclass' and 'Subclass' columns. Found: {list(df.columns)}"
            )

        col_super = cols["superclass"]
        col_sub   = cols["subclass"]

        # 2) Build relationships (remove blanks / NaN)
        children_map = defaultdict(list)
        parents = {}
        all_nodes = set()

        def _is_blank(x):
            if x is None:
                return True
            if isinstance(x, float) and math.isnan(x):
                return True
            s = str(x).strip()
            return s == "" or s.lower() == "nan"

        clean_rows = 0
        for _, row in df.iterrows():
            superclass = str(row[col_super]).strip()
            subclass   = str(row[col_sub]).strip()
            if _is_blank(superclass) or _is_blank(subclass):
                continue
            children_map[superclass].append(subclass)
            parents[subclass] = superclass
            all_nodes.add(superclass)
            all_nodes.add(subclass)
            clean_rows += 1

        print(f"Valid relationships (after cleanup): {clean_rows}")
        print(f"Total nodes: {len(all_nodes)}")

        # 3) Root nodes (nodes that have no parent)
        true_roots = [n for n in all_nodes if n not in parents]
        print(f"Root nodes ({len(true_roots)}): {true_roots[:10]}{' ...' if len(true_roots) > 10 else ''}")

        # 4) Check for missing parents
        print("\n=== Data Validation ===")
        orphan_nodes = []
        for child, parent in parents.items():
            if parent not in all_nodes:
                orphan_nodes.append((child, parent))
        if orphan_nodes:
            print(f"Missing parent relationships: {len(orphan_nodes)} (showing top 10):")
            for child, missing_parent in orphan_nodes[:10]:
                print(f"  {child} -> {missing_parent} (missing)")
        else:
            print("No missing parent relationships found")

        # 5) Function: get full path from root to a given node
        def get_full_path_to_root(node):
            path = [node]
            current = node
            visited = set()
            while current in parents and current not in visited:
                visited.add(current)
                current = parents[current]
                path.append(current)
            return path[::-1]

        # 6) Leaf nodes (nodes with no children)
        leaf_nodes = [
            n for n in all_nodes 
            if (n not in children_map or len(children_map[n]) == 0)
        ]
        print(f"\nLeaf nodes: {len(leaf_nodes)}")

        # 7) Collect all leaf-to-root paths and remove duplicates
        all_paths = [get_full_path_to_root(leaf) for leaf in leaf_nodes]
        unique_paths = []
        seen = set()
        for p in all_paths:
            t = tuple(p)
            if t not in seen:
                seen.add(t)
                unique_paths.append(p)

        print(f"Unique paths collected: {len(unique_paths)}")

        if not unique_paths:
            print("No hierarchical paths found.")
            return None

        # 8) Normalize depth and create final DataFrame
        max_depth = max(len(p) for p in unique_paths)
        rows = []
        for p in unique_paths:
            padded = p + [None] * (max_depth - len(p))
            rows.append(padded)

        df_expanded = pd.DataFrame(rows, columns=[f"Level_{i+1}" for i in range(max_depth)])

        # 9) Preview
        print("\n=== Preview ===")
        print(f"Generated DataFrame: {len(df_expanded)} rows x {df_expanded.shape[1]} columns")
        print(df_expanded.head(10).to_string(index=False))

        # 10) Save CSV (UTF-8 BOM for Excel compatibility)
        df_expanded.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
        print(f"\nCSV saved: {output_csv_path}")

        return df_expanded

    except Exception as e:
        print(f"Error during processing: {e}")
        import traceback
        traceback.print_exc()
        return None


# ========= Export OWL & TTL from Level_* hierarchy CSV =========
def export_levels_csv_to_owl(level_csv_path: str, output_owl_path: str,
                             base_iri: str = "http://example.org/dilon#"):
    """
    Convert a Level_1..Level_N CSV into an OWL ontology (RDF/XML),
    and also export a Turtle (.ttl) version.
    - Each row represents a root → leaf path
    - Each adjacent pair is converted into an rdfs:subClassOf triple
    - Class labels are preserved as rdfs:label
    - IRIs are sanitized to be safe and unique
    """
    print(f"\nStarting OWL export: {level_csv_path} -> {output_owl_path}")
    df = pd.read_csv(level_csv_path)

    # Identify Level_* columns
    level_cols = [c for c in df.columns if re.match(r"^Level_\d+$", str(c))]
    level_cols.sort(key=lambda s: int(re.findall(r"\d+", s)[0]))
    if not level_cols:
        raise ValueError("No Level_* columns found (expected Level_1, Level_2, ...).")

    # Sanitize local IRI names
    def sanitize(name: str) -> str:
        s = re.sub(r"\s+", "_", str(name).strip())
        s = re.sub(r"[^\w\u00A0-\uFFFF]", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
        if not s:
            s = "Class"
        if re.match(r"^\d", s):
            s = "_" + s
        return s

    label_to_local = {}
    local_to_label = {}

    def local_name(label: str) -> str:
        base = sanitize(label)
        cand = base
        i = 2
        while cand in local_to_label and local_to_label[cand] != label:
            cand = f"{base}_{i}"
            i += 1
        local_to_label[cand] = label
        label_to_local[label] = cand
        return cand

    edges = set()
    nodes = set()

    # Extract nodes and edges from hierarchy
    for _, row in df.iterrows():
        vals = []
        for c in level_cols:
            v = row.get(c, None)
            if pd.isna(v) or (isinstance(v, str) and v.strip() == ""):
                break
            vals.append(str(v).strip())
        for i, label in enumerate(vals):
            nodes.add(local_name(label))
            if i > 0:
                child = local_name(vals[i])
                parent = local_name(vals[i - 1])
                edges.add((child, parent))

    doc_iri = base_iri.rstrip("#")

    def iri(local: str) -> str:
        return f"{base_iri}{local}"

    def xml_escape(s: str) -> str:
        return (
            str(s)
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace('"', "&quot;")
        )

    # Build RDF/XML as text
    lines = []
    lines.append('<?xml version="1.0" encoding="UTF-8"?>')
    lines.append('<rdf:RDF')
    lines.append('  xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"')
    lines.append('  xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#"')
    lines.append('  xmlns:owl="http://www.w3.org/2002/07/owl#"')
    lines.append(f'  xml:base="{doc_iri}">')
    lines.append("")
    lines.append(f'  <owl:Ontology rdf:about="{doc_iri}"/>')
    lines.append("")

    for local, label in sorted(local_to_label.items()):
        lines.append(f'  <owl:Class rdf:about="{iri(local)}">')
        lines.append(f'    <rdfs:label>{xml_escape(label)}</rdfs:label>')
        for (child, parent) in edges:
            if child == local:
                lines.append(f'    <rdfs:subClassOf rdf:resource="{iri(parent)}"/>')
        lines.append("  </owl:Class>")

    lines.append("</rdf:RDF>")

    # Save RDF/XML (.owl)
    owl_xml = "\n".join(lines)
    Path(output_owl_path).write_text(owl_xml, encoding="utf-8")
    print(f"OWL export complete: {output_owl_path}")

    # Also save Turtle (.ttl) using rdflib
    ttl_path = str(output_owl_path).replace(".owl", ".ttl")
    g = Graph()
    g.parse(data=owl_xml, format="xml")
    g.serialize(ttl_path, format="turtle")
    print(f"Turtle export complete: {ttl_path}")

    print(f"Summary: {len(nodes)} classes, {len(edges)} subClassOf axioms\n")
    return {
        "classes": len(nodes),
        "subClassOf": len(edges),
        "owl": output_owl_path,
        "ttl": ttl_path,
    }


# ===== Execution =====
if __name__ == "__main__":
    base_path = "/Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2"
    input_csv = f"{base_path}/DILON_merged.csv"                        # (Superclass, Subclass)
    output_levels_csv = f"{base_path}/DILON_merged_with_hierarchy.csv" # (Level_1..N)
    output_owl = f"{base_path}/DILON_merged_with_hierarchy.owl"        # Final OWL output

    print("Starting expanded DILON hierarchy generation!")
    result_df = expand_dilon_hierarchy_fixed(input_csv, output_levels_csv)

    if result_df is not None:
        print("\n→ Exporting as OWL and TTL")
        export_levels_csv_to_owl(
            level_csv_path=output_levels_csv,
            output_owl_path=output_owl,
            base_iri="http://example.org/dilon#",  # modify if needed
        )
        print("Task completed!")
    else:
        print("Task failed!")


Starting expanded DILON hierarchy generation!
Input file: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_merged.csv
Output file: /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_merged_with_hierarchy.csv
CSV loaded successfully: 491 relationships
Valid relationships (after cleanup): 491
Total nodes: 489
Root nodes (1): ['Dietary_ontology']

=== Data Validation ===
No missing parent relationships found

Leaf nodes: 352
Unique paths collected: 352

=== Preview ===
Generated DataFrame: 352 rows x 7 columns
         Level_1              Level_2                  Level_3                Level_4                       Level_5 Level_6 Level_7
Dietary_ontology           Descriptor                Qualifier                Pattern                       Regular    None    None
Dietary_ontology      Personal_factor         Physical_finding        Allergy_symptom                          Rash    None    None
Dietary_ontology Food_handling_method Foo

# 4. Merge annotations from DILON v0.2 into the updated hierarchical ontology

In [ ]:
#!/usr/bin/env python
# coding: utf-8

from rdflib import Graph, RDF, RDFS
from rdflib.namespace import OWL, SKOS, DCTERMS
from pathlib import Path

# === 1) File paths ===
base = Path("/Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2")
path_v02       = base / "DILONv0.2.ttl"                       # Original ontology (with annotations)
path_original  = base / "DILON_merged_with_hierarchy.ttl"     # Updated hierarchy version

# Output locations (must be writable)
out_target_ttl = base / "DILON_annotation_merged_original.ttl"
out_target_owl = base / "DILON_annotation_merged_original.owl"  # Optional: also save RDF/XML

def ensure_exists(p: Path, name: str):
    """Raise error if file does not exist."""
    if not p.exists():
        raise FileNotFoundError(f"{name} file not found: {p.as_posix()}")

def guess_format(p: Path):
    """Infer RDF format from file extension."""
    ext = p.suffix.lower()
    if ext == ".ttl":
        return "turtle"
    if ext in (".owl", ".rdf", ".xml"):
        return "xml"
    return None

# Prepare
ensure_exists(path_v02,  "DILONv0.2")
ensure_exists(path_original, "DILON_merged_with_hierarchy")
out_target_ttl.parent.mkdir(parents=True, exist_ok=True)

# Load RDF graph
def load_graph(path: Path) -> Graph:
    g = Graph()
    fmt = guess_format(path)
    g.parse(path.as_posix(), format=fmt) if fmt else g.parse(path.as_posix())
    return g

g_v02     = load_graph(path_v02)
g_original = load_graph(path_original)

# Local name extraction
def localname(uri):
    s = str(uri)
    return s.split("#")[-1] if "#" in s else s.rstrip("/").split("/")[-1]

# Index classes
old_classes = {}
for c in set(g_v02.subjects(RDF.type, OWL.Class)) | set(g_v02.subjects(RDF.type, RDFS.Class)):
    old_classes[localname(c)] = c

new_classes = {}
for c in set(g_original.subjects(RDF.type, OWL.Class)) | set(g_original.subjects(RDF.type, RDFS.Class)):
    new_classes[localname(c)] = c

# Collect annotation properties from v0.2
annotation_props = set(g_v02.subjects(RDF.type, OWL.AnnotationProperty))
annotation_like_preds = {
    RDFS.label, RDFS.comment, SKOS.prefLabel, SKOS.altLabel, DCTERMS.description
} | annotation_props

# Merge annotations
copied = 0
mapped = 0

for lname, new_cls in new_classes.items():
    old_cls = old_classes.get(lname)
    if not old_cls:
        continue
    mapped += 1

    for p, o in g_v02.predicate_objects(old_cls):
        if p in annotation_like_preds and (new_cls, p, o) not in g_original:
            g_original.add((new_cls, p, o))
            copied += 1

print(f"[OK] Number of matched classes: {mapped}")
print(f"[OK] Number of added annotation triples: {copied}")

# === Save outputs ===
g_original.serialize(out_target_ttl.as_posix(), format="turtle")
g_original.serialize(out_target_owl.as_posix(), format="xml")  # optional RDF/XML version

print(f"[SAVE] {out_target_ttl.as_posix()}")
print(f"[SAVE] {out_target_owl.as_posix()}")


[OK] Number of matched classes: 244
[OK] Number of added annotation triples: 929
[SAVE] /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_annotation_merged_original.ttl
[SAVE] /Users/jinsunjung/Desktop/Ontology expansion/Validation/Validation_2/DILON_annotation_merged_original.owl
